<a href="https://colab.research.google.com/github/alyairawan02/publicdiscussion_analysis/blob/main/SMOTE_%26_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from imblearn.over_sampling import SMOTE

In [ ]:
import random

# Fungsi augmentasi sederhana
def augment_text(text):
    if isinstance(text, str):
        words = text.split()
        if len(words) > 1:
            random.shuffle(words)
        return ' '.join(words)
    return text

In [ ]:
#augmentasi data positif dan negatif

import pandas as pd
# 1. Baca data awal
df = pd.read_excel('hasil_labeling_3.xlsx')

# 2. Augmentasi POSITIVE
pos_data = df[df['sent_label'].str.upper() == 'POSITIVE']
aug_pos = pos_data.sample(n=2 * len(pos_data), replace=True, random_state=42).copy()
aug_pos['cleaning'] = aug_pos['cleaning'].apply(augment_text)
aug_pos['sent_label'] = 'POSITIVE'
df_aug = pd.concat([df, aug_pos], ignore_index=True)

# 3. Augmentasi NEGATIVE dari data yang sudah diperluas
neg_data = df_aug[df_aug['sent_label'].str.upper() == 'NEGATIVE']
aug_neg = neg_data.sample(n=(1500 - len(neg_data)), replace=True, random_state=42).copy()
aug_neg['cleaning'] = aug_neg['cleaning'].apply(augment_text)
aug_neg['sent_label'] = 'NEGATIVE'

# 4. Gabung semua jadi satu
df_final = pd.concat([df_aug, aug_neg], ignore_index=True)

# 5. Cek hasil distribusi
print(df_final['sent_label'].value_counts())
print(f"\n✅ Augmentasi lengkap berhasil. Jumlah data akhir: {len(df_final)}")

sent_label
POSITIVE    1572
NEGATIVE    1500
NEUTRAL     1038
Name: count, dtype: int64

✅ Augmentasi lengkap berhasil. Jumlah data akhir: 4110


In [ ]:
print(df_final['sent_label'].value_counts())

sent_label
POSITIVE    1572
NEGATIVE    1500
NEUTRAL     1038
Name: count, dtype: int64


In [ ]:
!pip install transformers torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# Load model IndoBERT Sentiment Analysis
model_name = "crypter70/IndoBERT-Sentiment-Analysis"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Buat pipeline untuk analisis sentimen
sentiment_pipe = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

Device set to use cpu


In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


# Encode label
label_encoder = LabelEncoder()
df_final['encoded_label'] = label_encoder.fit_transform(df_final['sent_label'])

In [ ]:
# Membersihkan kolom 'cleaning' dari NaN atau baris kosong
df_final = df_final[df_final['stemming_data'].notna()]
df_final = df_final[df_final['stemming_data'].str.strip() != '']
df_final.loc[:, 'stemming_data'] = df_final['stemming_data'].astype(str)
df_final = df_final.reset_index(drop=True)

In [ ]:
df_final['stemming_data'] = df_final['stemming_data'].fillna('').astype(str)
df_final.to_excel('df_final_with_encoded_label.xlsx', index=False)

In [ ]:
df_final = pd.read_excel('df_final_with_encoded_label.xlsx')

In [ ]:
#tokenizing

# Tokenisasi teks
tokenizer = Tokenizer(num_words=5000, oov_token='<OOV>')
tokenizer.fit_on_texts(df_final['stemming_data'])

sequences = tokenizer.texts_to_sequences(df_final['stemming_data'])

In [ ]:
# Padding
max_len = 50  #  distribusi panjang teks
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')

In [ ]:
print(df['stemming_data'].isnull().sum())

2


In [ ]:
df = df.dropna(subset=['stemming_data', 'sent_label'])

In [ ]:
X_text = df['stemming_data']
y_labels = df['sent_label']

# Ubah teks ke bentuk fitur numerik yang bisa diterima SVM
vectorizer = TfidfVectorizer(max_features=7000, ngram_range=(1,2))  # Boleh disesuaikan
X_tfidf = vectorizer.fit_transform(X_text)

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

# SMOTE hanya untuk data train
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

svm_model = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_model.fit(X_train_smote, y_train_smote)

y_pred = svm_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.7354260089686099
Classification Report:
               precision    recall  f1-score   support

    NEGATIVE       0.66      0.80      0.73       134
     NEUTRAL       0.75      0.83      0.79       208
    POSITIVE       0.87      0.46      0.60       104

    accuracy                           0.74       446
   macro avg       0.76      0.70      0.71       446
weighted avg       0.75      0.74      0.73       446

Confusion Matrix:
 [[107  25   2]
 [ 30 173   5]
 [ 24  32  48]]
